# Prove A Deployment Worked Without Fooling Yourself

Companion notebook for the [Measuring Deployment Success](https://codemia.io/courses/forward_deployed_engineering/fde_measuring_deployment_success) lesson on Codemia.

**What you will build**: the analysis behind a quarterly value report. A claims triage system
rolls out region by region, and the sponsor wants to know how much it cut the backlog. You will
simulate twenty six weeks of claims data with a seasonal dip that has nothing to do with the
system, show how a simple before and after comparison takes credit for that dip, estimate the
effect properly with difference in differences using regions that have not gone live yet, check
that adoption leads the outcome, track whether users keep using the system and turn the result
into a conservative value figure with a range the customer finance team can check.

No GPU. `numpy` and `matplotlib` only.

## Setup

In [ ]:
%pip install -q numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(5)
WEEKS = 26

## Concept recap

The lagging outcome is the backlog: days a claim waits for a decision. Many things move it besides
the system: seasons, staffing, a new product line. A before and after comparison in one region
credits all of them to the deployment. A staggered rollout gives a better comparison for free: in
any given week, regions that have not gone live yet show what would have happened anyway.
Difference in differences compares the change in live regions with the change in waiting regions
over the same weeks, so shared movements cancel out.

The leading measure is adoption: the share of eligible claims that go through the system. If the
chain from adoption to outcome holds, adoption should move first and the backlog should follow.

## Simulate a staggered rollout

In [ ]:
REGIONS = {'north': 6, 'south': 9, 'west': 12, 'east': 18, 'central': 99}   # go live week, 99 = not this half
TRUE_EFFECT = -4.0          # days of backlog removed at full adoption

weeks = np.arange(WEEKS)
season = -2.5 * np.clip((weeks - 8) / 6, 0, 1)        # a seasonal dip from week 8 onward
data = {}
for r, live in REGIONS.items():
    level = rng.uniform(18, 22)
    weeks_live = np.clip(weeks - live, 0, None)
    adoption = np.where(weeks >= live, 0.85 * (1 - np.exp(-weeks_live / 3)), 0.0)
    # the backlog responds to adoption with a delay of about three weeks
    lagged = np.concatenate([np.zeros(3), adoption[:-3]])
    backlog = level + season + TRUE_EFFECT * lagged + rng.normal(0, 0.5, WEEKS)
    data[r] = {'live': live, 'adoption': adoption, 'backlog': backlog}

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6))
for r, d in data.items():
    a1.plot(weeks, d['adoption'], label=r)
    a2.plot(weeks, d['backlog'], label=r)
a1.set_title('Adoption: share of claims through the system'); a1.set_xlabel('week')
a2.set_title('Backlog in days'); a2.set_xlabel('week'); a2.legend(fontsize=7)
plt.tight_layout()
plt.show()

Every region dips from week eight, including central, which never went live. That dip is the
season. Any method that credits it to the system will overstate the value.

## The naive before and after

In [ ]:
d = data['north']
before = d['backlog'][:d['live']].mean()
after = d['backlog'][-6:].mean()
print(f'north: backlog {before:.1f} days before go live, {after:.1f} days in the last six weeks')
print(f'naive effect {after - before:+.1f} days   (true effect at full adoption {TRUE_EFFECT * 0.85:+.1f})')

The naive figure includes the seasonal dip. A sponsor who reports it will be embarrassed next
quarter when the season reverses and the backlog creeps back up with the system still running.

## Difference in differences with regions not yet live

For each week, compare live regions with regions that are not live that week, measured as the
change from each region's own pre go live average. Waiting regions absorb the season; what is
left is the effect. Here we estimate it for weeks at least six weeks after each region went live,
when adoption is near its plateau.

In [ ]:
def did(data, min_weeks_live=6):
    effects = []
    for w in weeks:
        treated, control = [], []
        for r, d in data.items():
            base = d['backlog'][:min(d['live'], 6)].mean()     # each region's own early baseline
            change = d['backlog'][w] - base
            if w >= d['live'] + min_weeks_live:
                treated.append(change)
            elif w < d['live']:
                control.append(change)
        if treated and control:
            effects.append(np.mean(treated) - np.mean(control))
    return np.array(effects)

eff = did(data)
print(f'weeks with both groups: {len(eff)}   effect {eff.mean():+.1f} days')

# A range: resample regions with replacement and recompute.
names = list(data)
boots = []
for _ in range(1000):
    pick = rng.choice(names, len(names), replace=True)
    sample = {f'{n}_{i}': data[n] for i, n in enumerate(pick)}
    e = did(sample)
    if len(e):
        boots.append(e.mean())
lo, hi = np.quantile(boots, [0.05, 0.95])
print(f'90% range {lo:+.1f} to {hi:+.1f} days')

The estimate sits close to the true effect, and the naive figure sits well outside the range.
The range comes from resampling only five regions, so treat it as rough and say so in the report
rather than quoting one number with false precision. Holding a region back for a few more weeks
keeps a control group longer, which is a rollout decision worth raising with the sponsor before
it is made.

## Does adoption lead the outcome?

If the chain holds, the backlog should follow adoption after a delay. Correlate the weekly change
in adoption with the weekly change in backlog at different lags, pooled across live regions.

In [ ]:
def lag_corr(lag):
    xs, ys = [], []
    for d in data.values():
        if d['live'] >= WEEKS:
            continue
        da = np.diff(d['adoption'])
        db = np.diff(d['backlog'])
        if lag > 0:
            xs.append(da[:-lag]); ys.append(db[lag:])
        else:
            xs.append(da); ys.append(db)
    return np.corrcoef(np.concatenate(xs), np.concatenate(ys))[0, 1]

lags = range(0, 7)
cors = [lag_corr(l) for l in lags]
for l, c in zip(lags, cors):
    print(f'lag {l} weeks: correlation {c:+.2f}')
print('strongest at', int(np.argmin(cors)), 'weeks')

The correlation is strongest, and negative, about three weeks after adoption changes: more
claims through the system now, a shorter backlog a few weeks later. That lag is what lets the
team report adoption in the first month as evidence that the outcome is coming, and it tells the
sponsor when to expect it.

## Are users staying?

Adoption at launch is easy; adoption in month three is the real test. Simulate user activity and
build a cohort table: of the users active in their first month, what share are still active later?

In [ ]:
N_USERS = 300
start = rng.integers(0, 3, N_USERS)               # month each user first used the system
drop = rng.random(N_USERS)
months = 6
active = np.zeros((N_USERS, months), dtype=bool)
for u in range(N_USERS):
    p = 0.95
    for m in range(start[u], months):
        # a workflow fix in month 3 lowers the monthly drop off from 15 to 4 percent
        p *= 0.85 if m < 3 else 0.96
        active[u, m] = drop[u] < p

print('cohort   ' + ' '.join(f'm{k + 1:<4}' for k in range(months)))
for c in range(3):
    m = start == c
    row = [active[m, k].mean() if k >= c else np.nan for k in range(months)]
    print(f'month {c + 1}  ' + ' '.join('  -  ' if np.isnan(v) else f'{v:4.0%} ' for v in row))

Read across a row to follow one cohort. Retention falls fast at first and then flattens after the
workflow fix in month three, which is exactly the pattern the sponsor needs to see: early users
who stayed are the ones whose hours count in the value figure.

## A conservative value figure

Turn the backlog effect into money the customer finance team will accept. Count only active users,
use the conservative end of the effect range and include every cost, including the customer's own
staff time. Show the range, not just one number.

In [ ]:
CLAIMS_PER_WEEK = 4200          # across the live regions
COST_PER_CLAIM_DAY = 1.10       # holding cost and customer contacts per claim per day of backlog, agreed with finance
ACTIVE_SHARE = active[:, -1].mean()
COSTS = {'licence': 180_000, 'infrastructure': 40_000, 'customer staff time': 75_000, 'training': 20_000}

def annual_value(effect_days):
    # Each claim decided a day sooner saves one claim day of holding cost. Scale by the share of
    # users still active, so only retained use earns credit.
    return -effect_days * CLAIMS_PER_WEEK * 52 * COST_PER_CLAIM_DAY * ACTIVE_SHARE

total_cost = sum(COSTS.values())
for label, e in (('conservative end', hi), ('central estimate', eff.mean()), ('naive figure', after - before)):
    v = annual_value(e)
    print(f'{label:<17} effect {e:+.1f} days -> value {v:>9,.0f} a year   return {v / total_cost:.1f}x   payback {12 * total_cost / max(v, 1):.0f} months')
print(f'\nactive share used: {ACTIVE_SHARE:.0%}   total annual cost: {total_cost:,}')

The naive figure would have promised a much larger return. Present the conservative end as the
headline, the central estimate as the expected value and the method on one page, then ask the
customer analyst to check the inputs. A figure the customer has checked is one they will defend.

## Exercises

1. Move central's go live to week 20 so every region eventually goes live. How does the
   difference in differences estimate change in the last weeks, when no region is left waiting?
2. Make the effect depend on adoption without any lag. What happens to the lag correlation, and
   what would that tell you about the chain in a real deployment?
3. Remove the workflow fix from the retention simulation and recompute the value figure. How much
   of the return depended on users staying?
4. Add a second, unrelated change in week 14 that affects only south, such as a staffing increase.
   Which estimate absorbs it, and how would you spot it from the data?

## What to take away

- Before and after comparisons credit everything that changed to the system. Do not report them.
- A staggered rollout gives a control group. Difference in differences uses it.
- Few regions means a wide range. Report the range and plan the rollout to narrow it.
- Adoption should lead the outcome by a delay you can measure. Use it to set expectations early.
- Value counts only real, retained use, uses the conservative end and includes every cost.